# YOLO26x-Pose Evaluation | Keypoint Detection | Stomata (4 KP)

**Model**: YOLO26x-Pose  (finetuned)
**Task**: Keypoint Detection
**Eval**: COCO OKS-based Keypoint AP + Bbox AP

This notebook reproduces the YOLO26X-Pose evaluation reported in the CVPRW stomata keypoint benchmark.
It is intended for AP-based benchmark evaluation only.
Some public benchmark splits contain annotations only; for those splits, users must separately download the original images from the source links listed in the Hugging Face dataset card.

---
## 1. Configuration & Environment

In [1]:
import sys
import platform
import warnings
import logging
import json
from pathlib import Path
from typing import Dict, List, Tuple
from collections import defaultdict

import numpy as np
import torch
import cv2
from tqdm.notebook import tqdm

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import pandas as pd
from ultralytics import YOLO

warnings.filterwarnings('ignore')

print('=' * 60)
print('ENVIRONMENT')
print('=' * 60)
print(f'Python:       {sys.version.split()[0]}')
print(f'PyTorch:      {torch.__version__}')
print(f'CUDA:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'OpenCV:       {cv2.__version__}')
import ultralytics; print(f'Ultralytics:  {ultralytics.__version__}')
from importlib.metadata import version as pkg_version
print(f'pycocotools:  {pkg_version("pycocotools")}')

ENVIRONMENT
Python:       3.12.2
PyTorch:      2.7.0+cu126
CUDA:         NVIDIA A100 80GB PCIe
OpenCV:       4.9.0
Ultralytics:  8.4.14
pycocotools:  2.0.10


In [ ]:
# ======================== CONFIGURATION ========================
# Add the dataset and model files from the hugging face links in your custom directory
# Some benchmark splits in the public dataset release contain annotations only.
# For those splits, users must separately download the original images from the source links listed in the hugging face dataset card 
# Place them in the expected folder structure before running evaluation.


from pathlib import Path
import os
import numpy as np
import torch

MODEL_NAME = "YOLO26X-Pose"
TASK = "keypoint_detection"
YOLO_WEIGHTS = Path(os.environ.get("YOLO_WEIGHTS", "weights/YOLO26X-Pose/best.pt"))
#https://huggingface.co/Sainath001/stomata-keypoint-benchmark-cvpr-agrivision-2026-models/tree/main/YOLO26X-Pose
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "data/stomata-keypoint-benchmark/datasets"))
#https://huggingface.co/datasets/Sainath001/stomata-keypoint-benchmark-cvpr-agrivision-2026-Dataset/tree/main/datasets
SCORE_THRESHOLD = 0.3
MAX_DETS_PER_IMAGE = 300
STOMATA_CAT_ID = 1
NUM_KEYPOINTS = 4
OKS_SIGMAS = np.array([0.1, 0.1, 0.1, 0.1], dtype=np.float32)

def _has_images(d: Path) -> bool:
    exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
    for ext in exts:
        if any(d.glob(f"*{ext}")):
            return True
    return False

def build_datasets(root: str) -> dict:
    root = Path(root)
    datasets = {}

    for ds_dir in sorted(root.iterdir()):
        if not ds_dir.is_dir():
            continue
        gt_json = ds_dir / "coco.json"
        if not gt_json.exists():
            jsons = list(ds_dir.glob("*.json"))
            if len(jsons) == 1:
                gt_json = jsons[0]
            else:
                continue
        images_dir = ds_dir / "images"
        if images_dir.is_dir() and _has_images(images_dir):
            images_path = images_dir
        else:
            images_path = ds_dir
        key = ds_dir.name
        datasets[key] = {
            "gt_json": str(gt_json),
            "images": str(images_path),
        }
    return datasets

DATASETS = build_datasets(DATA_ROOT)

RESULTS_ROOT = Path("artifacts") / MODEL_NAME
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

PRED_ROOT = RESULTS_ROOT / "preds"
PRED_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Model:           {MODEL_NAME}")
print(f"Task:            {TASK}")
print(f"Score threshold: {SCORE_THRESHOLD}")
print(f"Max dets/image:  {MAX_DETS_PER_IMAGE}")
print(f"Num keypoints:   {NUM_KEYPOINTS}")
print(f"OKS sigmas:      {OKS_SIGMAS}")
print(f"Datasets:        {list(DATASETS.keys())}")
print(f"YOLO weights:    {YOLO_WEIGHTS}")
print(f"Pred dir:        {PRED_ROOT}")
print(f"Device:          {DEVICE}")

---
## 2. Generate COCO-format Keypoint Predictions

In [3]:
# ======================== UTILITY FUNCTIONS ========================

def xyxy_to_xywh(box_xyxy: np.ndarray) -> List[float]:
    """Convert [x1, y1, x2, y2] to COCO [x, y, w, h]."""
    x1, y1, x2, y2 = [float(v) for v in box_xyxy]
    return [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)]


def clip_xywh(x, y, w, h, W, H):
    """Clip box [x, y, w, h] to image bounds."""
    x = max(0.0, min(float(x), W - 1.0))
    y = max(0.0, min(float(y), H - 1.0))
    w = max(0.0, min(float(w), W - x))
    h = max(0.0, min(float(h), H - y))
    return x, y, w, h


def load_coco_items(gt_json, images_root):
    """Load COCO GT and return (coco_obj, items_list, size_map)."""
    coco = COCO(gt_json)
    imgs = coco.loadImgs(coco.getImgIds())
    items = []
    size_map = {}
    images_root = Path(images_root)
    for im in imgs:
        iid = int(im['id'])
        H, W = int(im['height']), int(im['width'])
        fp = images_root / im['file_name']
        if not fp.exists():
            fp = images_root / Path(im['file_name']).name
        if fp.exists():
            items.append((iid, str(fp), (H, W)))
            size_map[iid] = (H, W)
    return coco, items, size_map

In [ ]:
# ======================== YOLO POSE INFERENCE ========================

def infer_yolo_pose(
    weights_path: str,
    items: List[Tuple[int, str, Tuple[int, int]]],
    device: str = 'cuda:0',
    score_thr: float = 0.3,
    max_dets: int = 300,
    num_kp: int = 4,
) -> List[Dict]:
    """
    Run YOLO-pose inference and return COCO-format keypoint predictions.
    
    Each prediction dict contains:
      - image_id, category_id, bbox [x,y,w,h], score
      - keypoints [x1,y1,v1, x2,y2,v2, ...] (v=2 if conf>0, else v=0)
    """
    model = YOLO(weights_path)
    print(f'Model loaded: {weights_path}')
    
    preds = []
    for image_id, path, (H, W) in tqdm(items, desc='YOLO-Pose Inference'):
        img = cv2.imread(path)
        if img is None:
            continue
        
        results = model.predict(
            source=img,
            conf=score_thr,
            iou=0.7,
            device=device,
            max_det=max_dets,
            verbose=False,
        )
        
        if len(results) == 0:
            continue
        r = results[0]
        
        # Check boxes exist
        if r.boxes is None or r.boxes.xyxy is None:
            continue
        
        boxes_xyxy = r.boxes.xyxy.cpu().numpy()
        scores = r.boxes.conf.cpu().numpy()
        
        # Get keypoints: shape (N, num_kp, 3) where last dim is (x, y, conf)
        if r.keypoints is not None and r.keypoints.data is not None:
            kps_data = r.keypoints.data.cpu().numpy()  # (N, K, 3)
        else:
            continue
        
        for i, (box_xyxy, score) in enumerate(zip(boxes_xyxy, scores)):
            s = float(score)
            if s < score_thr:
                continue
            
            # Convert box
            bbox_xywh = xyxy_to_xywh(box_xyxy)
            x, y, w, h = clip_xywh(*bbox_xywh, W, H)
            if w <= 0 or h <= 0:
                continue
            
            # Convert keypoints to COCO format: [x1,y1,v1, x2,y2,v2, ...]
            kp = kps_data[i]  # (K, 3) — x, y, conf
            coco_kp = []
            for ki in range(num_kp):
                kx, ky, kc = float(kp[ki, 0]), float(kp[ki, 1]), float(kp[ki, 2])
                # visibility: 0=not labeled, 1=labeled but occluded, 2=labeled and visible
                v = 2 if kc > 0 else 0
                coco_kp.extend([kx, ky, v])
            
            preds.append({
                'image_id': int(image_id),
                'category_id': STOMATA_CAT_ID,
                'bbox': [x, y, w, h],
                'score': s,
                'keypoints': coco_kp,
            })
    
    print(f'Total predictions: {len(preds)}')
    return preds

In [ ]:
# ======================== RUN INFERENCE ON ALL SPLITS ========================

pred_files = {}  # {split_name: pred_json_path}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n{'=' * 60}")
    print(f'Dataset: {ds_name}')
    print(f"{'=' * 60}")
    
    out_json = PRED_ROOT / f'{ds_name}_kp_preds.json'
    
    # Skip if predictions already exist
    if out_json.exists():
        print(f'  Predictions already exist: {out_json}')
        print(f'  Delete file to regenerate.')
        pred_files[ds_name] = str(out_json)
        continue
    
    # Load GT
    coco_gt, items, size_map = load_coco_items(ds_cfg['gt_json'], ds_cfg['images'])
    print(f'  Images found: {len(items)}')
    
    # Run inference
    preds = infer_yolo_pose(
        weights_path=YOLO_WEIGHTS,
        items=items,
        device=DEVICE,
        score_thr=SCORE_THRESHOLD,
        max_dets=MAX_DETS_PER_IMAGE,
        num_kp=NUM_KEYPOINTS,
    )
    
    # Save
    with open(out_json, 'w') as f:
        json.dump(preds, f, indent=2)
    print(f'  Saved {len(preds)} predictions → {out_json}')
    pred_files[ds_name] = str(out_json)

print(f"\n{'=' * 60}")
print('PREDICTION FILES:')
for name, path in pred_files.items():
    print(f'  {name}: {path}')

---
## 3. COCO Keypoint Evaluation (OKS-based AP)

In [6]:
# ======================== KEYPOINT EVALUATION ========================

def run_coco_keypoint_eval(
    gt_json: str,
    pred_json: str,
    sigmas: np.ndarray = None,
) -> Dict:
    """
    Run COCO keypoint evaluation using OKS.
    
    Args:
        gt_json:   Path to COCO ground truth JSON (with keypoint annotations)
        pred_json: Path to COCO prediction JSON (with keypoints)
        sigmas:    OKS sigma per keypoint (controls tolerance)
    
    Returns:
        Dict with AP, AP50, AP75, AR metrics
    """
    coco_gt = COCO(gt_json)
    if 'info' not in coco_gt.dataset:
        coco_gt.dataset['info'] = {} 
    coco_dt = coco_gt.loadRes(pred_json)
    
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='keypoints')
    
    # Override default COCO sigmas with our stomata-specific sigmas
    if sigmas is not None:
        coco_eval.params.kpt_oks_sigmas = sigmas
    
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    
    s = coco_eval.stats
    return {
        'KP_AP':       s[0],
        'KP_AP50':     s[1],
        'KP_AP75':     s[2],
        'KP_AP_M':     s[3],
        'KP_AP_L':     s[4],
        'KP_AR':       s[5],
        'KP_AR50':     s[6],
        'KP_AR75':     s[7],
        'KP_AR_M':     s[8],
        'KP_AR_L':     s[9],
    }

In [7]:
# ======================== RUN KEYPOINT EVAL ========================

kp_results = {}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n{'=' * 60}")
    print(f'Keypoint Eval: {ds_name}')
    print(f"{'=' * 60}")
    
    pred_json = pred_files.get(ds_name)
    if pred_json is None or not Path(pred_json).exists():
        print(f'  Prediction file not found!')
        continue
    
    try:
        metrics = run_coco_keypoint_eval(
            gt_json=ds_cfg['gt_json'],
            pred_json=pred_json,
            sigmas=OKS_SIGMAS,
        )
        kp_results[ds_name] = metrics
    except Exception as e:
        print(f'  Error: {e}')
        import traceback; traceback.print_exc()

# Display
if kp_results:
    print(f"\n{'=' * 60}")
    print(f'KEYPOINT AP RESULTS — {MODEL_NAME} (OKS sigmas={OKS_SIGMAS[0]})')
    print(f"{'=' * 60}")
    df_kp = pd.DataFrame(kp_results).T
    df_kp = (df_kp * 100).round(2)
    display(df_kp)


Keypoint Eval: T_HR5MZ
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.42s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.000
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50

,KP_AP,KP_AP50,KP_AP75,KP_AP_M,KP_AP_L,KP_AR,KP_AR50,KP_AR75,KP_AR_M,KP_AR_L
T_HR5MZ,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
T_HR5WH,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
T_MSAA,7.88,18.37,5.68,7.88,-100.00,13.49,26.80,11.94,13.49,-100.00
T_MZA,33.72,50.64,38.99,33.72,-100.00,37.89,51.02,43.87,37.89,-100.00
T_MZLP,30.61,49.44,35.13,30.61,50.00,34.58,50.47,40.12,34.57,50.00
T_NKBY,0.00,0.00,0.00,-100.00,0.00,0.00,0.00,0.00,-100.00,0.00
T_SBGH,0.00,0.00,0.00,0.00,-100.00,0.00,0.00,0.00,0.00,-100.00
T_TCAB,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
T_TCMZ,0.04,0.10,0.00,0.00,0.04,0.15,0.37,0.00,0.00,0.16


---
## 4. COCO Bbox Evaluation

In [8]:
# ======================== BBOX EVALUATION ========================

def run_coco_bbox_eval(gt_json: str, pred_json: str) -> Dict:
    """Run standard COCO bbox evaluation."""
    coco_gt = COCO(gt_json)
    if 'info' not in coco_gt.dataset:
        coco_gt.dataset['info'] = {} 
    coco_dt = coco_gt.loadRes(pred_json)
    
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    
    s = coco_eval.stats
    return {
        'Box_AP':     s[0],
        'Box_AP50':   s[1],
        'Box_AP75':   s[2],
        'Box_AP_S':   s[3],
        'Box_AP_M':   s[4],
        'Box_AP_L':   s[5],
        'Box_AR@1':   s[6],
        'Box_AR@10':  s[7],
        'Box_AR@100': s[8],
    }


bbox_results = {}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n{'=' * 60}")
    print(f'Bbox Eval: {ds_name}')
    print(f"{'=' * 60}")
    
    pred_json = pred_files.get(ds_name)
    if pred_json is None or not Path(pred_json).exists():
        print(f'  Prediction file not found!')
        continue
    
    try:
        metrics = run_coco_bbox_eval(
            gt_json=ds_cfg['gt_json'],
            pred_json=pred_json,
        )
        bbox_results[ds_name] = metrics
    except Exception as e:
        print(f'  Error: {e}')

# Display
if bbox_results:
    print(f"\n{'=' * 60}")
    print(f'BBOX AP RESULTS — {MODEL_NAME}')
    print(f"{'=' * 60}")
    df_box = pd.DataFrame(bbox_results).T
    df_box = (df_box * 100).round(2)
    display(df_box)


Bbox Eval: T_HR5MZ
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.87s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.564
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.953
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.628
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.562
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.602
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.008
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.073
 Average Recall     (AR) @[ IoU=0.50:0.95 | 

,Box_AP,Box_AP50,Box_AP75,Box_AP_S,Box_AP_M,Box_AP_L,Box_AR@1,Box_AR@10,Box_AR@100
T_HR5MZ,56.38,95.29,62.77,-100.0,56.19,60.20,0.76,7.29,63.45
T_HR5WH,1.95,6.50,0.40,-100.0,2.99,2.03,0.92,2.75,2.75
T_MSAA,16.01,40.90,6.12,-100.0,16.03,-100.00,1.26,11.22,22.03
T_MZA,48.71,91.62,44.98,-100.0,48.71,-100.00,1.65,16.21,58.23
T_MZLP,42.89,89.89,30.93,-100.0,42.89,20.00,1.51,15.45,51.61
T_NKBY,0.00,0.00,0.00,-100.0,-100.00,0.00,0.00,0.00,0.00
T_SBGH,0.00,0.00,0.00,0.0,0.00,-100.00,0.00,0.00,0.00
T_TCAB,0.03,0.07,0.00,-100.0,0.30,0.00,0.37,0.37,0.37
T_TCMZ,1.79,3.12,1.87,-100.0,0.00,1.96,1.67,3.56,3.56


---
## 5. Summary

In [9]:
# ======================== COMBINED SUMMARY ========================

print(f"\n{'=' * 60}")
print(f'SUMMARY — {MODEL_NAME}')
print(f'Confidence threshold: {SCORE_THRESHOLD}')
print(f'OKS sigmas: {OKS_SIGMAS.tolist()}')
print(f"{'=' * 60}")

summary_rows = []
for ds_name in DATASETS:
    row = {'Split': ds_name}
    if ds_name in kp_results:
        row.update(kp_results[ds_name])
    if ds_name in bbox_results:
        row.update(bbox_results[ds_name])
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index('Split')

# Show key columns
key_cols = ['Box_AP', 'Box_AP50', 'KP_AP', 'KP_AP50', 'KP_AP75', 'KP_AR']
key_cols = [c for c in key_cols if c in df_summary.columns]
df_key = (df_summary[key_cols] * 100).round(2)

print('\nKey Metrics (%):')
display(df_key)


SUMMARY — YOLO26x-Pose_Non_clahe
Confidence threshold: 0.3
OKS sigmas: [0.10000000149011612, 0.10000000149011612, 0.10000000149011612, 0.10000000149011612]

Key Metrics (%):


,Box_AP,Box_AP50,KP_AP,KP_AP50,KP_AP75,KP_AR
Split,,,,,,
T_HR5MZ,56.38,95.29,0.00,0.00,0.00,0.00
T_HR5WH,1.95,6.50,0.00,0.00,0.00,0.00
T_MSAA,16.01,40.90,7.88,18.37,5.68,13.49
T_MZA,48.71,91.62,33.72,50.64,38.99,37.89
T_MZLP,42.89,89.89,30.61,49.44,35.13,34.58
T_NKBY,0.00,0.00,0.00,0.00,0.00,0.00
T_SBGH,0.00,0.00,0.00,0.00,0.00,0.00
T_TCAB,0.03,0.07,0.00,0.00,0.00,0.00
T_TCMZ,1.79,3.12,0.04,0.10,0.00,0.15


In [ ]:
# ======================== SAVE ALL RESULTS ========================

results_output = {
    'model': MODEL_NAME,
    'task': TASK,
    'score_threshold': SCORE_THRESHOLD,
    'oks_sigmas': OKS_SIGMAS.tolist(),
    'num_keypoints': NUM_KEYPOINTS,
    'keypoint_eval': kp_results,
    'bbox_eval': bbox_results,
}

results_json = PRED_ROOT / 'evaluation_results.json'
with open(results_json, 'w') as f:
    json.dump(results_output, f, indent=2, default=float)
print(f'Results saved to: {results_json}')

Expected local layout
data/stomata-keypoint-benchmark/
├── KP-Train/
├── T-MZA/
├── T-MZLP/
├── ...
weights/YOLO26X-Pose/
└── best.pt